# DDL: `dbspend360_pool_cloud_cost_explorer`

Per-pool cloud VM cost, populated by the cost explorer ETL's pool-tag query.
On **AWS** this is `aws_cloud_cost_explorer_app.run_pool()` (plan
`plan_pool_pipeline_ec2_cost.md` §4.2 / CP5); on **Azure** it is
`azure_cloud_cost_explorer_app.run_pool()` (plan
`plan_pool_pipeline_azure_cost.md` §4.2 / AZ-CP2). This is a **shared,
cloud-agnostic** table — both providers write the same grain into it; no
provider-specific fork.

Idle/warm pool capacity is tagged `DatabricksInstancePoolId`, **not**
`ClusterId` (the same Databricks tagging behavior on both clouds), so it is
invisible to `dbspend360_cloud_cost_explorer` (which groups only by
`ClusterId`). This dedicated table is the primary home of pool VM cost. The
explorer nets out any cost that *also* carries a `ClusterId` (§4.3 guard), so
pool and cluster cloud cost are disjoint — the tabs are additive, not
overlapping.

Grain: `(instance_pool_id, cost_incurred_date, currency)`, a single
`cloud_cost` bucket on **both** clouds; the `compute/storage/network/other`
segments are reserved but `NULL`. On AWS that bucket is the sum of the EC2
family. On Azure the segments are NULL **by design**: Azure caps Cost
Management grouping at 2 items and both slots are spent on the pool + cluster
tags for the netting guard, leaving no slot for `MeterCategory` — so
disjointness is prioritized over segmentation (plan
`plan_pool_pipeline_azure_cost.md` §4.3). `idle_cloud_cost` /
`active_cloud_cost` are reserved for the future `instance_events`-based split
(§4.5) and stay `NULL` until that fast-follow lands.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_pool_cloud_cost_explorer (
  instance_pool_id    STRING,
  cloud_cost          DOUBLE,
  compute_cost        DOUBLE,   -- NULL on AWS & Azure (single cloud_cost bucket)
  storage_cost        DOUBLE,   -- NULL on AWS & Azure
  network_cost        DOUBLE,   -- NULL on AWS & Azure
  other_cost          DOUBLE,   -- NULL on AWS & Azure
  idle_cloud_cost     DOUBLE,   -- reserved (instance_events split, plan 4.5); NULL until then
  active_cloud_cost   DOUBLE,   -- reserved (instance_events split, plan 4.5); NULL until then
  currency            STRING,
  created_at          TIMESTAMP,
  updated_at          TIMESTAMP,
  cost_incurred_date  DATE
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_pool_cloud_cost_explorer")